In [15]:
# pip install tensorflow pillow numpy matplotlib


In [16]:
import tensorflow as tf
import numpy as np
from PIL import Image
import pickle
from langchain_ollama import OllamaLLM

# Load model
model = tf.keras.models.load_model('ingredients_classification_model.h5')

# Load class labels
with open('class_labels.pkl', 'rb') as f:
    class_labels = pickle.load(f)


In [17]:
from PIL import Image
import os

# Function to predict ingredients in an image using sliding window
def predict_ingredients(image_path, patch_size=224, step_size=112, threshold=0.5):

    # Args:
    #     patch_size: Size of the sliding window (default: 224x224).
    #     step_size: How much to move the window at each step.
    #     threshold: Probability threshold to accept a label.

    image = Image.open(image_path).convert('RGB')
    width, height = image.size

    found_labels = set()

    # Slide window
    for y in range(0, height - patch_size + 1, step_size):
        for x in range(0, width - patch_size + 1, step_size):
            patch = image.crop((x, y, x + patch_size, y + patch_size))

            # Preprocess patch
            patch_array = tf.keras.preprocessing.image.img_to_array(patch)
            patch_array = tf.keras.applications.mobilenet_v2.preprocess_input(patch_array)
            patch_array = np.expand_dims(patch_array, axis=0)

            # Predict
            prediction = model.predict(patch_array, verbose=0)
            predicted_index = np.argmax(prediction)
            confidence = prediction[0][predicted_index]

            if confidence >= threshold:
                label = class_labels[predicted_index]
                found_labels.add(label)
    return list(found_labels)

In [18]:
image_path = 'input_image/own_image.jpg'
predicted_ingredients = predict_ingredients(image_path, step_size=200, threshold=0.5)

# Print results and image path
print("Predicted ingredients:", predicted_ingredients)
print("Image path:", image_path)

Predicted ingredients: ['Orange', 'Avocado', 'Arla-Ecological-Medium-Fat-Milk', 'Satsumas', 'God-Morgon-Orange-Red-Grapefruit-Juice', 'Oatly-Oat-Milk', 'Papaya', 'Banana', 'Solid-Potato', 'God-Morgon-Orange-Juice', 'Regular-Tomato', 'God-Morgon-Red-Grapefruit-Juice', 'Leek', 'Cucumber', 'Red-Beet', 'Floury-Potato', 'Arla-Medium-Fat-Milk']
Image path: input_image/own_image.jpg


In [19]:
# Ask the servings
servings = str(input("\nFor how many people will this be? (1-10): "))
# Ask important features
important = str(input("\nIs there anything important that you want to add before the recipe gets generated?\n\n(e.g. Allergies, Prefered ways, difficulty level, duration...)\nIf you don't have anything just press \"Enter\""))

In [20]:
# Initialize llama3 LLM
llmModel = OllamaLLM(model="llama3")

# Handle important features
if important.strip():
    important = 'Important feature: ' + important
else:
    important = ''

# Handle serving amount
if servings.strip() == '' or servings:
    servings = ' is for 2 people'
else:
    try:
        servings_int = int(servings)
        servings_str = f' is for {servings_int} people'
    except ValueError:
        servings_str = ' is for 2 people'

# Generate prompt
prompt=f"Make a recipe, only including the title, servings{servings}, with these ingredients: {predicted_ingredients}. {important}"
print(prompt)
result = llmModel.invoke(input=prompt)
print(result)

Make a recipe, only including the title, servings is for 2 people, with these ingredients: ['Orange', 'Avocado', 'Arla-Ecological-Medium-Fat-Milk', 'Satsumas', 'God-Morgon-Orange-Red-Grapefruit-Juice', 'Oatly-Oat-Milk', 'Papaya', 'Banana', 'Solid-Potato', 'God-Morgon-Orange-Juice', 'Regular-Tomato', 'God-Morgon-Red-Grapefruit-Juice', 'Leek', 'Cucumber', 'Red-Beet', 'Floury-Potato', 'Arla-Medium-Fat-Milk']. 
Here is a recipe that uses all the ingredients:

**Tropical Avocado Soup for 2**

 Servings: 2 people

Ingredients:

* 1 Orange, peeled and segmented
* 1/2 ripe Avocado, diced
* 1 cup Arla-Ecological-Medium-Fat-Milk
* 1 Satsuma, peeled and segmented
* 2 tbsp God-Morgon-Orange-Red-Grapefruit-Juice
* 1/4 cup Oatly-Oat-Milk
* 1/2 Papaya, diced
* 1 Banana, sliced
* 2 Solid-Potato, peeled and diced
* 2 tbsp God-Morgon-Orange-Juice
* 1 Regular-Tomato, seeded and chopped
* 1 Leek, chopped
* 1 Cucumber, peeled and thinly sliced
* 1 Red-Beet, peeled and thinly sliced
* 2 Floury-Potato, peele